## Project NLP and Deep Learning

### 1. Project proposal presentation

In the presentation, you have 5 minutes to present your research proposal. During the presentation, you should explain:

* What is the topic of your project, what is the current state of this topic/task/setup
* What is the new part of your project
* What is the research question of your project

We have proposed a number of topics in the slides which can be found on LearnIt, you can either pick one of these or come up with your own. If you pick your own, we suggest to get a pre-approval with Rob van der Goot.

**Deadline for uploading slides: day before the presentation (23:59)**  (pdf only, they will be put into one long pdf for a smooth presentation)

### 2. Baseline
To get your project started, you start with implementing a baseline model. Ideally, this is going to be the main baseline that you are going to compare to in your paper. Note that this baseline should be more advanced than just predicting the majority class (O).

We will use EWT portion of the [Universal NER project](http://www.universalner.org/), which we provide with this notebook for convenience. You can use the train data (`en_ewt-ud-train.iob2`) and dev data(`en_ewt-ud-dev.iob2`) to build your baseline, then upload your prediction on the test data (`en_ewt-ud-test.iob2`).

It is important to upload your predictions in same format as the training and dev files, so that the `span_f1.py` script can be used.

Note that you do not have to implement your baseline from scratch, you can use for example the code from the RNN or BERT assignments as a starting point.

**Deadline: 20-03 on LearnIt (14:00)**

In [ ]:
!pip install torch
!pip install transformers
!pip install datasets
!pip install tqdm
!pip install evaluate
!pip install seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
! gdown --id 1TrgWxk42wPorYeQRjZtIod8nBQ2SqXPP
! gdown --id 1tnwydaunyAvJwB2UtosYGcV_lpsvSoZz
! gdown --id 1PZijwpMc2LQzKZLsWbDlBMXgaANUKnH_

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1TrgWxk42wPorYeQRjZtIod8nBQ2SqXPP
To: /content/en_ewt-ud-dev.iob2
100% 620k/620k [00:00<00:00, 84.9MB/s]
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1tnwydaunyAvJwB2UtosYGcV_lpsvSoZz
To: /content/en_ewt-ud-test-masked.iob2
100% 617k/617k [00:00<00:00, 79.1MB/s]
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...


In [28]:
import torch
import random
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments, set_seed, AutoConfig)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import evaluate

# Set seed for reproducibility
set_seed(42)

# Define constants
MODEL_NAME = 'bert-base-multilingual-uncased' # "bert-base-uncased"
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5

# Load tokenizer
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load dataset using load_dataset
data_files = {
    "train": "/content/en_ewt-ud-train.iob2",
    "validation": "/content/en_ewt-ud-dev.iob2",
    "test": "/content/en_ewt-ud-test-masked.iob2"
}

# Function to parse IOB2 dataset and extract label list
def parse_iob2(file_path):
    sentences, labels = [], []
    words, tags = [], []
    unique_labels = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):  # Skip metadata lines
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
                continue
            parts = line.split()  # Split by whitespace
            if len(parts) >= 2:
                words.append(parts[1])  # First column: token
                tags.append(parts[2])  # Second column: label
                unique_labels.add(parts[2])
        if words:
            sentences.append(words)
            labels.append(tags)

    return {"tokens": sentences, "ner_tags": labels}, sorted(unique_labels)


dataset_dict = {}
all_labels = set()
for split, file in data_files.items():
    parsed_data, labels = parse_iob2(file)
    dataset_dict[split] = parsed_data
    all_labels.update(labels)

# Convert label list to mapping
label_list = sorted(all_labels)  # Ensure consistent order
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# Convert labels to integers
def convert_labels(dataset):
    dataset["ner_tags"] = [[label2id[tag] for tag in tags] for tags in dataset["ner_tags"]]
    return dataset

dataset_dict = {split: convert_labels(dataset) for split, dataset in dataset_dict.items()}
raw_datasets = DatasetDict({
    split: Dataset.from_dict(dataset_dict[split]) for split in dataset_dict
})




c:\Users\user\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
def tokenize_and_align_labels(examples):
    # Tokenize the text
    tokenized_inputs = tokenizer(
        examples["tokens"],  # Assuming "tokens" is a list of words already split
        max_length=128,
        padding=True,  # Ensures consistent length
        truncation=True,
        is_split_into_words=True,  # Important if you already have tokenized words
        return_tensors="pt"  # Returns PyTorch tensors (adjust as needed)
    )

    all_labels = []
    for batch_index, labels in enumerate(examples["ner_tags"]):  # Assuming ner_tags are already integers
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        label_ids = []
        prev_word_id = None

        # Iterate over the word_ids for token alignment
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # Special tokens get -100 (e.g., [CLS], [SEP])
            elif word_id == prev_word_id:
                label_ids.append(-100)  # Subword tokens get -100 (tokens from a split word)
            else:
                label_ids.append(labels[word_id])  # Use the actual label for the word

            prev_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels  # Add labels to tokenized inputs
    return tokenized_inputs

# Apply tokenization
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [ ]:
label2id

{'B-LOC': 0,
 'B-ORG': 1,
 'B-PER': 2,
 'I-LOC': 3,
 'I-ORG': 4,
 'I-PER': 5,
 'O': 6}

In [ ]:
def tokenize_and_align_labels(examples, label_all_tokens=True):
    """
    Function to tokenize and align labels with respect to the tokens. This function is specifically designed for
    Named Entity Recognition (NER) tasks where alignment of the labels is necessary after tokenization.

    Parameters:
    examples (dict): A dictionary containing the tokens and the corresponding NER tags.
                     - "tokens": list of words in a sentence.
                     - "ner_tags": list of corresponding entity tags for each word.

    label_all_tokens (bool): A flag to indicate whether all tokens should have labels.
                             If False, only the first token of a word will have a label,
                             the other tokens (subwords) corresponding to the same word will be assigned -100.

    Returns:
    tokenized_inputs (dict): A dictionary containing the tokenized inputs and the corresponding labels aligned with the tokens.
    """
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        # word_ids() => Return a list mapping the tokens
        # to their actual word in the initial sentence.
        # It Returns a list indicating the word corresponding to each token.
        previous_word_idx = None
        label_ids = []
        # Special tokens like `<s>` and `<\s>` are originally mapped to None
        # We need to set the label to -100 so they are automatically ignored in the loss function.
        for word_idx in word_ids:
            if word_idx is None:
                # set –100 as the label for these special tokens
                label_ids.append(-100)
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            elif word_idx != previous_word_idx:
                # if current word_idx is != prev then its the most regular case
                # and add the corresponding token
                label_ids.append(label[word_idx])
            else:
                # to take care of sub-words which have the same word_idx
                # set -100 as well for them, but only if label_all_tokens == False
                label_ids.append(label[word_idx] if label_all_tokens else -100)
                # mask the subword representations after the first subword

            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [ ]:
# Map the tokenize_and_align_labels function to the raw datasets

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]

# Inspect a few training samples after tokenization
for index in random.sample(range(len(train_dataset)), 3):
    print(f"Sample {index} of the training set: {train_dataset[index]}")

Sample 12149 of the training set: {'tokens': ['Good', '.'], 'ner_tags': [6, 6], 'input_ids': [101, 12050, 119, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1], 'labels': [-100, 6, 6, -100]}
Sample 4506 of the training set: {'tokens': ['Gregg'], 'ner_tags': [2], 'input_ids': [101, 68618, 102], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1], 'labels': [-100, 2, -100]}
Sample 4012 of the training set: {'tokens': ['Thanks', '!'], 'ner_tags': [6, 6], 'input_ids': [101, 47530, 106, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1], 'labels': [-100, 6, 6, -100]}


In [ ]:

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
data_collator = DataCollatorForTokenClassification(tokenizer)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)

small_eval_dataset = tokenized_datasets["validation"].select(range(200))  # Smaller eval set
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=6,
    weight_decay=0.001,
)

# Define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train model
trainer.train()

# Evaluate model
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    return metric.compute(predictions=predictions, references=labels)

trainer.evaluate(eval_dataset=tokenized_datasets["validation"], metric_key_prefix="eval")

model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-10-3d558c681e6f>:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to h

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: diko4ev (diko4ev-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# **✅ Save the trained model and tokenizer**
SAVE_PATH = "./trained_model"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Model and tokenizer saved to {SAVE_PATH}")

✅ Model and tokenizer saved to ./trained_model


In [ ]:
from transformers import AutoModelForTokenClassification
import torch
import json
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments
import torch

data_collator = DataCollatorForTokenClassification(tokenizer)

# Load trained model
model = AutoModelForTokenClassification.from_pretrained("./trained_model")
model.eval()

# Initialize the Trainer
# Define training arguments (you can customize this)
training_args = TrainingArguments(
    output_dir="./results",  # where to store the results
    per_device_eval_batch_size=BATCH_SIZE,  # batch size during evaluation
    evaluation_strategy="epoch",  # how often to evaluate
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Run inference on the test dataset
predictions_test = trainer.predict(tokenized_datasets["test"])
predictions_val = trainer.predict(tokenized_datasets["validation"])

# Extract label predictions
pred_labels_val = torch.argmax(torch.tensor(predictions_val.predictions), dim=2).numpy()
pred_labels_test = torch.argmax(torch.tensor(predictions_test.predictions), dim=2).numpy()
# Convert predicted IDs to text labels
final_predictions_val = [
    [id2label[p] for p in sentence] for sentence in pred_labels_val
]

final_predictions_test = [
    [id2label[p] for p in sentence] for sentence in pred_labels_test
]

HFValidationError: Repo id must use alphanumeric chars or '-', '_', '.', '--' and '..' are forbidden, '-' and '.' cannot start or end the name, max length is 96: './trained_model'.

In [ ]:
# Convert predictions to IOB format
iob_predictions = []

for sent_idx, final_pred in enumerate(final_predictions_test):
    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions.append(sentence_iob)

# Save predictions to a file in IOB format
with open("ner_predictions.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

In [ ]:
# Convert predictions to IOB format
iob_predictions = []

for sent_idx, final_pred in enumerate(final_predictions_val):

    tokens = tokenized_datasets["validation"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["validation"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions.append(sentence_iob)

# Save predictions to a file in IOB format
with open("ner_predictions_dev.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

In [ ]:
# Set seed for reproducibility

# Define constants
MODEL_NAME = 'bert-base-multilingual-cased' # "bert-base-uncased"
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# Load tokenizer
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
data_collator = DataCollatorForTokenClassification(tokenizer)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)

small_eval_dataset = tokenized_datasets["validation"].select(range(200))  # Smaller eval set
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=6,
    weight_decay=0.001,
)

# Define trainer
trainer_m_c = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train model
trainer_m_c.train()

# Evaluate model
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    return metric.compute(predictions=predictions, references=labels)

trainer_m_c.evaluate(eval_dataset=tokenized_datasets["validation"], metric_key_prefix="eval")


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-15-8399355aa718>:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_m_c = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: diko4ev (diko4ev-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.243900,0.190425
2,0.117700,0.168421
3,0.080800,0.177660
4,0.045500,0.199673
5,0.032100,0.208353
6,0.020500,0.221287


{'eval_loss': 0.22128674387931824,
 'eval_runtime': 10.5498,
 'eval_samples_per_second': 189.671,
 'eval_steps_per_second': 11.943,
 'epoch': 6.0}

In [ ]:
# Run inference on the test dataset
predictions_test = trainer_m_c.predict(tokenized_datasets["test"])
predictions_val = trainer_m_c.predict(tokenized_datasets["validation"])

# Extract label predictions
pred_labels_val = torch.argmax(torch.tensor(predictions_val.predictions), dim=2).numpy()
pred_labels_test = torch.argmax(torch.tensor(predictions_test.predictions), dim=2).numpy()
# Convert predicted IDs to text labels
final_predictions_val = [
    [id2label[p] for p in sentence] for sentence in pred_labels_val
]

final_predictions_test = [
    [id2label[p] for p in sentence] for sentence in pred_labels_test
]

In [ ]:
# Convert predictions to IOB format
iob_predictions_t = []
iob_predictions_v = []

for sent_idx, final_pred in enumerate(final_predictions_test):
    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions_t.append(sentence_iob)
for sent_idx, final_pred in enumerate(final_predictions_val):
    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions_v.append(sentence_iob)

# Save predictions to a file in IOB format
with open("mbc_predictions.iob", "w") as f:
    for sentence in iob_predictions_t:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line
with open("mbc_predictions_dev.iob", "w") as f:
    for sentence in iob_predictions_v:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

In [ ]:
MODEL_NAME = 'bert-base-cased' # "bert-base-uncased"
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# Load tokenizer
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
data_collator = DataCollatorForTokenClassification(tokenizer)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)

#small_eval_dataset = tokenized_datasets["validation"].select(range(200))  # Smaller eval set
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.001,
    report_to="none"
)

# Define trainer
trainer_c = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train model
trainer_c.train()

# Evaluate model
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    return metric.compute(predictions=predictions, references=labels)

trainer_c.evaluate(eval_dataset=tokenized_datasets["validation"], metric_key_prefix="eval")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-6-8adbd6b809e5>:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_c = Trainer(


Epoch,Training Loss,Validation Loss
1,0.091800,0.056316
2,0.019400,0.057485
3,0.011700,0.062074


{'eval_loss': 0.06207415834069252,
 'eval_runtime': 12.2151,
 'eval_samples_per_second': 163.814,
 'eval_steps_per_second': 10.315,
 'epoch': 3.0}

In [ ]:
# Run inference on the test dataset
predictions_test = trainer_c.predict(tokenized_datasets["test"])
predictions_val = trainer_c.predict(tokenized_datasets["validation"])

# Extract label predictions
pred_labels_val = torch.argmax(torch.tensor(predictions_val.predictions), dim=2).numpy()
pred_labels_test = torch.argmax(torch.tensor(predictions_test.predictions), dim=2).numpy()
# Convert predicted IDs to text labels
final_predictions_val = [
    [id2label[p] for p in sentence] for sentence in pred_labels_val
]

final_predictions_test = [
    [id2label[p] for p in sentence] for sentence in pred_labels_test
]

In [ ]:
# Convert predictions to IOB format
iob_predictions_t = []
iob_predictions_v = []

for sent_idx, final_pred in enumerate(final_predictions_test):
    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions_t.append(sentence_iob)
for sent_idx, final_pred in enumerate(final_predictions_val):
    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions_v.append(sentence_iob)

# Save predictions to a file in IOB format
with open("bbc_predictions.iob", "w") as f:
    for sentence in iob_predictions_t:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line
with open("bbc_predictions_dev.iob", "w") as f:
    for sentence in iob_predictions_v:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

### 3. Project proposal

The written proposal should consist of maximum one page in [ACL-format](https://github.com/acl-org/acl-style-files) (The bibliography does not count for the word limit). In here, you should explain the last three points from the list above and place your project in a larger context (previous work).

Make sure your proposal is:
* Novel to some extent
* Doable within the time-frame

*hint* The [ACL Anthology](https://aclanthology.org/) contains almost all peer-reviewed NLP papers.

**Deadline: 03-04 on LearnIt (14:00)**

### 4. Final project
The final project has a maximum size of 5 pages (excluding bibliography and appendix), using the [ACL style files](https://github.com/acl-org/acl-style-files)

Besides the main paper (discussed in class), you have to include:
* Group contributions. State who was responsible for which part of the project. Here you may state if there
were any serious unequal workloads among group members. This should be put in the appendix.
* A report on usage of chatbots. We follow: https://2023.aclweb.org/blog/ACL-2023-policy/
   * Add a section in appendix if you made use of a chatbot (since we do not use a Responsible NLP Checklist)
   * Include each stage on the ACL policy, and indicate to what extent you used a chatbot
   * Use with care!, you are responsible for the project and plagiarism, correctness etc.

You can also put additional results and details in the appendix. However, the paper itself should be standalone, and understandable without consulting the appendix.

Furthermore, the code should be available on www.github.itu.dk (with a link in a footnote at the end of the abstract) , it should include a README with instructions on how to reproduce your results.

**Deadline: 23-05 on LearnIt** Please check the checklist below before uploading!

Optionally, you can upload a draft a week before **16-05 (before 09:00)** for an extra round of feedback

## Analysis

Analysis is essential for the interpretation of your results. In this section we will shortly describe some different types of analysis. We strongly suggest to use at least one of these:

* **Ablation study**: Leave out a certain part of the model, to study its effects. For example, disable the tokenizer, remove a certain (group of) feature(s), or disable the stop-word removal. If the performance drops a lot, it means that this part of the model contributes heavily to the models final performance. This is commonly done in 1 table, while disabling different parts of the model. Note that you can also do this the other way around, i.e. use only one feature (group) at a time, and test performance
* **Learning curve**: Evaluate how much data your model needs to reach a certain performance. Especially for the data augmentation projects this is essential.
* **Quantitative analysis**: Automated means of analyzing in which cases your model performs worse. This can for example be done with a confusion matrix.
* **Qualitative analysis**: Manually inspect a certain number of errors, and try to categorize them/find trends. Can be combined with the quantitative analysis, i.e., inspect 100 cases of positive reviews predicted to be negative and 100 cases of negative reviews predicted to be positive
* **Feature importance**: In traditional machine learning methods, one can often extract and inspect the weights of the features. In sklearn these can be found in: `trained_model.coef_`
* **Other metrics**: per class scores, partial matches, or count how often the span-borders were correct, but the label wrong.
* **Input words importance**: To gain insight into which words have a impact on prediction performance (positive, negative), we can analyze per-word impact: given a trained model, replace a given word with
the unknown word token and observe the change in prediction score (probability for a class). This is
shown in Figure 4 of [Rethmeier et al (2018)](https://aclweb.org/anthology/W18-6246) (a paper on controversy detection), also shown below: red-colored
tokens were important for controversy detection, blue-colored token decreased prediction scores.

<img width=400px src=example.png>

Note that this is a non-exhaustive list, and you are encouraged to also explore additional analyses.

### Checklist final project
Please check all these items before handing in your final report. You only have to upload a pdf file on learnit, and make sure a link to the code is included in the report and the code is accesible.

* Are all group members and their email addresses specified?
* Does the group report include a representative project title?
* Does the group report contain an abstract?
* Does the introduction clearly specify the research intention and research question?
* Does the group report adequately refer to the relevant literature?
* Does the group report properly use figure, tables and examples?
* Does the group report provide and discuss the empirical results?
* Is the group report proofread?
* Does the pdf contain the link to the project’s github repo?
* Is the github repo accessible to the public (within ITU)?
* Is the group report maximum 5 pages long, excluding references and appendix?
* Are the group contributions added in the appendix?
* Does the repository contain all scripts and code to reproduce the results in the group report? Are instructions
 provided on how to run the code?
